In [11]:
!pip install easyocr jiwer pandas tqdm matplotlib seaborn

In [12]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import easyocr
from jiwer import wer

In [13]:
ROOT = Path("/kaggle/input/datasets/evgeniyashpirnova/trainvaldataset/all_data")

# Загружаем только validation
val_df = pd.read_csv(ROOT / "val" / "labels.csv")

val_df["image_path"] = val_df["filename"].apply(
    lambda x: str(ROOT / "val" / x)
)

dataset = val_df

## Расчет расстояния Левенштейна и метрики CER

### Расстояние Левенштейна

Расстояние Левенштейна характеризуется минимальным количеством операций, нужным, чтобы превратить одну строку в другую.

**Формула:**

$$
LD = S + D + I
$$

где:
S (Substitutions)— количество замен символов;

D (Deletions) — количество удалений символов;

I (Insertions) — количество вставленных лишних символов.

### Метрика CER

CER показывает, какую долю символов OCR распознал неправильно.

**Формула:**

$$
CER = \frac{LD}{N}
$$

где:
N— количество символов в эталонной строке (*Ground Truth*).

Чем меньше значение CER, тем лучше качество распознавания.

### Метрика WER

WER показывает долю слов, распознанных системой OCR неверно.

Для вычисления WER используется расстояние Левенштейна на уровне слов, а не отдельных символов.

Формула та же, что и в CER.


In [14]:
def levenshtein_distance(s1, s2):
    m, n = len(s1), len(s2)

    dp = [[0]*(n+1) for _ in range(m+1)]

    for i in range(m+1):
        dp[i][0] = i

    for j in range(n+1):
        dp[0][j] = j

    for i in range(1, m+1):
        for j in range(1, n+1):

            cost = 0 if s1[i-1] == s2[j-1] else 1

            dp[i][j] = min(
                dp[i-1][j] + 1,
                dp[i][j-1] + 1,
                dp[i-1][j-1] + cost
            )

    return dp[m][n]


def cer(gt, pred):
    if len(gt) == 0:
        return 0 if len(pred) == 0 else 1

    return levenshtein_distance(gt, pred) / len(gt)

### Объявление модели распознавания текста

In [15]:
reader = easyocr.Reader(
    ['de'],
    gpu=True,
    recog_network="best_accuracy",
    user_network_directory='custom_EasyOCR/model',
    model_storage_directory='custom_EasyOCR/user_network',
)

In [16]:
def recognize_text(image_path):
    result = reader.readtext(
        str(image_path),
        detail=0,
        paragraph=True
    )

    text = " ".join(result)

    return text.strip()

In [19]:
df = pd.read_csv("/kaggle/input/datasets/evgeniyashpirnova/trainvaldataset/all_data/val/labels.csv")

print(df.columns.tolist())
df.head()

['filename', 'words']


,filename,words
0,crop_000282.jpg,oxfordtube com
1,crop_002014.jpg,Franziusallee
2,crop_001719.jpg,Stolzeweg
3,crop_000994.jpg,traumhaft
4,crop_002033.jpg,Fahrschule


## Распознавание текста и сравнение с эталонным: оценка по метрикам cer, wer, accuracy (полное совпадение)

In [20]:
def evaluate_dataset(dataset):
    results = []

    for _, row in tqdm(dataset.iterrows(), total=len(dataset)):
        gt_text = str(row["words"])

        pred_text = recognize_text(row["image_path"])

        results.append({
            "filename": row["filename"],
            "gt": gt_text,
            "pred": pred_text,
            "cer": cer(gt_text, pred_text),
            "wer": wer(gt_text, pred_text),
            "exact_match": int(
                gt_text.strip() == pred_text.strip()
            )
        })

    return pd.DataFrame(results)

## Вывод метрик
В среднем около 31 символа из 100 распознаются неверно.

Чуть больше половины слов распознаются с ошибкой.

Только 57% изображений распознаны абсолютно без ошибок.

In [22]:
def calculate_metrics(df):
    return {
        "CER": df["cer"].mean(),
        "WER": df["wer"].mean(),
        "Accuracy": df["exact_match"].mean()
    }

val_metrics = calculate_metrics(val_results)

pd.DataFrame([val_metrics])

,CER,WER,Accuracy
0,0.310293,0.552726,0.570093


## Слова, которые были распознаны неправильно

In [25]:
errors = val_results[
    val_results["exact_match"] == 0
]

for _, row in errors.head(10).iterrows():
    print("FILE:")
    print(row["filename"])

    print("\nGT:")
    print(row["gt"])

    print("\nPRED:")
    print(row["pred"])

    print("\nCER:", round(row["cer"], 4))
    print("\nWER:", round(row["wer"], 4))

FILE:
crop_001719.jpg

GT:
Stolzeweg

PRED:
Stolzeweg_

CER: 0.1111

WER: 1.0
FILE:
crop_000994.jpg

GT:
traumhaft

PRED:
ge traumhaft

CER: 0.3333

WER: 1.0
FILE:
crop_002033.jpg

GT:
Fahrschule

PRED:
Fakrschule

CER: 0.1

WER: 1.0
FILE:
crop_000297.jpg

GT:
GEORGE

PRED:
GEORGE '

CER: 0.3333

WER: 1.0
FILE:
crop_001590.jpg

GT:
Zuwiderhandlumngen

PRED:
Jos Zuwiderhandluingenz

CER: 0.3333

WER: 2.0
FILE:
crop_000290.jpg

GT:
PLACE

PRED:
PUACE

CER: 0.2

WER: 1.0
FILE:
crop_001727.jpg

GT:
PIZZA

PRED:
PIZA

CER: 0.2

WER: 1.0
FILE:
crop_000507.jpg

GT:
Bücher

PRED:
Bucher

CER: 0.1667

WER: 1.0
FILE:
crop_001795.jpg

GT:
&

PRED:


CER: 1.0

WER: 1.0
FILE:
crop_000196.jpg

GT:
Comfort

PRED:
Comlort

CER: 0.1429

WER: 1.0
